# *DeBERTa-v3-base*

In [1]:
import os
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DebertaV2Tokenizer, DebertaV2ForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformer_utils import TextDataset, label_smoothing_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

LABEL2ID  = {'google': 0,
             'anthropic': 1,
             'meta': 2,
             'openai': 3,
             'human': 4}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
N_CLASSES = 5

device: cuda


## *dados*

In [ ]:
df_full  = pd.read_csv('../datasets/dataset_v2_full.csv',    sep=';')
df_ex    = pd.read_csv('../datasets/dataset-exemplos.csv',   sep=';')
df_subm1 = pd.read_csv('../Subm1/subm1_labels_revealed.csv', sep=';')
df_subm2 = pd.read_csv('../Subm2/subm2_labels_revealed.csv', sep=';')

def load_xy(df):
    df = df.dropna(subset=['Label'])
    df = df[df['Label'].str.strip().str.lower().isin(LABEL2ID.keys())]
    texts  = df['Text'].fillna('').tolist()
    labels = [LABEL2ID[l.strip().lower()] for l in df['Label'].tolist()]
    return texts, labels

texts_synth, y_synth = load_xy(df_full)
texts_r1,    y_r1    = load_xy(df_subm1)
texts_r2,    y_r2    = load_xy(df_subm2)
texts_real   = texts_r1 + texts_r2
y_real       = y_r1     + y_r2
texts_val,   y_val   = load_xy(df_ex)

cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_real + y_val)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)

print(f'sintéticos: {len(texts_synth)} | reais: {len(texts_real)} (subm1: {len(texts_r1)} + subm2: {len(texts_r2)})')
print(f'validação:  {len(texts_val)} exemplos do docente')
print('class weights:', {ID2LABEL[i]: f'{w:.2f}' for i, w in enumerate(cw)})

## *modelo e funções auxiliares*

In [3]:
from transformer_utils import TextDataset, label_smoothing_loss

MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LEN    = 128

tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)


def evaluate_model(model, loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds_all.extend(outputs.logits.argmax(dim=1).cpu().tolist())
            labels_all.extend(batch['labels'].tolist())

    acc = sum(p == t for p, t in zip(preds_all, labels_all)) / len(labels_all)
    f1  = f1_score(labels_all, preds_all, average='macro')
    return acc, f1, preds_all

print(f'modelo base: {MODEL_NAME}')

modelo base: microsoft/deberta-v3-base


## *optimização de hiperparâmetros (Optuna)*

In [4]:
import optuna

val_ds     = TextDataset(texts_val, y_val, tokenizer, MAX_LEN)
val_loader = DataLoader(val_ds, batch_size=32)
print(f'validação tokenizada: {len(val_ds)} amostras')


def train_one_trial(trial):

    lr           = trial.suggest_float('lr', 5e-6, 3e-5, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [16, 32])
    smoothing    = trial.suggest_float('label_smoothing', 0.05, 0.2)
    warmup_frac  = trial.suggest_float('warmup_frac', 0.05, 0.2)
    weight_decay = trial.suggest_float('weight_decay', 1e-3, 0.1, log=True)
    real_wt      = trial.suggest_int('real_weight', 5, 20)

    MAX_EPOCHS = 6
    PATIENCE   = 2

    t_train = texts_synth + texts_real * real_wt
    y_tr    = y_synth     + y_real     * real_wt
    train_ds     = TextDataset(t_train, y_tr, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = DebertaV2ForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES,
        id2label=ID2LABEL, label2id=LABEL2ID
    ).float().to(device)

    optimizer    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps  = len(train_loader) * MAX_EPOCHS
    warmup_steps = int(total_steps * warmup_frac)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_score = 0.0
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda', dtype=torch.bfloat16):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                              smoothing=smoothing, weights=class_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        val_acc, val_f1, _ = evaluate_model(model, val_loader)
        score = (val_f1 + val_acc) / 2

        trial.report(score, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if score > best_score:
            best_score = score
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    del model, optimizer, scheduler, train_ds, train_loader
    torch.cuda.empty_cache()

    return best_score


print('função de treino definida!')

validação tokenizada: 125 amostras
função de treino definida!


In [5]:
N_TRIALS = 10

study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=3),
    study_name='deberta-hparam-search'
)
study.optimize(train_one_trial, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nmelhor F1-macro: {study.best_value:.4f}')
print('melhores hiperparâmetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

[I 2026-04-01 12:07:29,260] A new study created in memory with name: deberta-hparam-search


  0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

[I 2026-04-01 12:37:16,184] Trial 0 finished with value: 0.6390215876253611 and parameters: {'lr': 2.8113480525723423e-05, 'batch_size': 32, 'label_smoothing': 0.18874223856952577, 'warmup_frac': 0.12492061299678299, 'weight_decay': 0.002475628927414196, 'real_weight': 16}. Best is trial 0 with value: 0.6390215876253611.


...

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

[I 2026-04-01 16:57:36,800] Trial 8 finished with value: 0.6787443769680612 and parameters: {'lr': 2.0625638698976323e-05, 'batch_size': 16, 'label_smoothing': 0.06403562265107055, 'warmup_frac': 0.14267720197982314, 'weight_decay': 0.022475497069200982, 'real_weight': 19}. Best is trial 6 with value: 0.6796745915374947.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

[I 2026-04-01 17:05:14,819] Trial 9 finished with value: 0.6290011532119535 and parameters: {'lr': 1.5731064080955456e-05, 'batch_size': 16, 'label_smoothing': 0.08645980347788652, 'warmup_frac': 0.1375959701730754, 'weight_decay': 0.010768163272434085, 'real_weight': 7}. Best is trial 6 with value: 0.6796745915374947.

melhor F1-macro: 0.6797
melhores hiperparâmetros:
  lr: 1.1478806169206716e-05
  batch_size: 16
  label_smoothing: 0.05955311467330515
  warmup_frac: 0.09739988292462173
  weight_decay: 0.09405826939621509
  real_weight: 14


## *treino final*

In [ ]:
LR           = study.best_params['lr']
BATCH_SIZE   = study.best_params['batch_size']
LABEL_SMOOTH = study.best_params['label_smoothing']
WARMUP_FRAC  = study.best_params['warmup_frac']
WEIGHT_DECAY = study.best_params['weight_decay']
REAL_WT      = study.best_params['real_weight']
MAX_EPOCHS   = 20
PATIENCE     = 5

t_train_final = texts_synth + texts_real * REAL_WT
y_train_final = y_synth     + y_real     * REAL_WT

train_ds     = TextDataset(t_train_final, y_train_final, tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print(f'treino final: {len(train_ds)} amostras | batch_size={BATCH_SIZE}')
print(f'lr={LR:.2e} | smoothing={LABEL_SMOOTH} | warmup={WARMUP_FRAC} | wd={WEIGHT_DECAY} | real_wt={REAL_WT}')

In [7]:
model = DebertaV2ForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=N_CLASSES,
    id2label=ID2LABEL, label2id=LABEL2ID
).float().to(device)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

best_f1    = 0.0
best_acc   = 0.0
best_state = None
no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss  = 0.0
    running_total = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=device.type == 'cuda', dtype=torch.bfloat16):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                          smoothing=LABEL_SMOOTH, weights=class_weights)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running_loss  += loss.item() * input_ids.size(0)
        running_total += input_ids.size(0)

    train_loss          = running_loss / running_total
    val_acc, val_f1, _ = evaluate_model(model, val_loader)

    marker = ' *' if val_f1 > best_f1 else ''
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}{marker}')

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_acc   = val_acc
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'early stopping na época {epoch}')
            break

model.load_state_dict(best_state)
print(f'\nmelhor modelo -> val_acc={best_acc:.4f} | val_f1={best_f1:.4f}')

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

Epoch 01/20 | train_loss=1.3823 | val_acc=0.5040 | val_f1=0.4247 *
Epoch 02/20 | train_loss=0.5532 | val_acc=0.5920 | val_f1=0.5122 *
Epoch 03/20 | train_loss=0.3693 | val_acc=0.6400 | val_f1=0.5330 *
Epoch 04/20 | train_loss=0.3574 | val_acc=0.5680 | val_f1=0.5150
Epoch 05/20 | train_loss=0.3436 | val_acc=0.6800 | val_f1=0.6034 *
Epoch 06/20 | train_loss=0.3431 | val_acc=0.6960 | val_f1=0.6060 *
Epoch 07/20 | train_loss=0.3446 | val_acc=0.5760 | val_f1=0.5250
Epoch 08/20 | train_loss=0.3453 | val_acc=0.5440 | val_f1=0.4561
Epoch 09/20 | train_loss=0.3398 | val_acc=0.6400 | val_f1=0.5457
Epoch 10/20 | train_loss=0.3412 | val_acc=0.6640 | val_f1=0.5951
Epoch 11/20 | train_loss=0.3389 | val_acc=0.6560 | val_f1=0.5795
early stopping na época 11

melhor modelo -> val_acc=0.6960 | val_f1=0.6060


In [8]:
SAVE_DIR = '../models/model_deberta'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'modelo guardado em {SAVE_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

modelo guardado em ../models/model_deberta


## *avaliação com dataset-exemplos*

In [ ]:
MODEL_DIR = '../models/model_deberta'
tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_DIR)
model     = DebertaV2ForSequenceClassification.from_pretrained(MODEL_DIR).float().to(device)
model.eval()

val_ds     = TextDataset(texts_val, y_val, tokenizer, max_len=128)
val_loader = DataLoader(val_ds, batch_size=64)

print(f'modelo carregado de {MODEL_DIR} | {len(texts_val)} exemplos de validação')

In [ ]:
val_acc, val_f1, val_preds = evaluate_model(model, val_loader)
print(f'[dataset-exemplos] accuracy={val_acc:.4f} | f1-macro={val_f1:.4f}')
print()
print(classification_report(
    [ID2LABEL[l] for l in y_val],
    [ID2LABEL[p] for p in val_preds],
    digits=3
))

## *resultados Optuna*

In [11]:
trials_df = study.trials_dataframe().sort_values('value', ascending=False)

cols = ['number', 'value', 'params_lr', 'params_batch_size',
        'params_label_smoothing', 'params_warmup_frac',
        'params_weight_decay', 'params_real_weight']
display_cols = [c for c in cols if c in trials_df.columns]

print('top 5 trials:')
print(trials_df[display_cols].head(5).to_string(index=False))

top 5 trials:
 number    value  params_lr  params_batch_size  params_label_smoothing  params_warmup_frac  params_weight_decay  params_real_weight
      6 0.679675   0.000011                 16                0.059553            0.097400             0.094058                  14
      8 0.678744   0.000021                 16                0.064036            0.142677             0.022475                  19
      1 0.655065   0.000007                 32                0.118958            0.088311             0.001129                  12
      7 0.653030   0.000008                 16                0.078105            0.083790             0.008639                  13
      5 0.653028   0.000010                 16                0.078721            0.178971             0.032191                  10
